# triangle-splatting :: Custom data :: Alaska_00

-----
- Conda env : [waikiki_statue](README.md#setup-a-conda-environment)
-----

### Check system

In [1]:
!nvidia-smi

Sat Oct  4 11:16:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 24%   44C    P5             35W /  250W |     591MiB /  11264MiB |     30%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download a video

In [2]:
import os
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)

In [3]:
import gdown
VIDEO_NAME = "Alaska_00"

FPS = 10
RES = 4
id = "1LCjjapYzev3hbvaiOY28EIU53OkEvNqt"
vid_path = f"./temp_data/{VIDEO_NAME}.mov"

gdown.download(id=id, output = vid_path)

Downloading...
From: https://drive.google.com/uc?id=1LCjjapYzev3hbvaiOY28EIU53OkEvNqt
To: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/triangle_splatting/temp_data/Alaska_00.mov
100%|██████████| 55.9M/55.9M [00:04<00:00, 11.3MB/s]


'./temp_data/Alaska_00.mov'

### Extract images from the video

In [4]:
DATASET_DIR_PATH = f"./temp_data/{VIDEO_NAME}"
IMAGES_DIR_PATH = os.path.join(DATASET_DIR_PATH, "images")
DATABASE_PATH = os.path.join(DATASET_DIR_PATH, "database.db")

Path(IMAGES_DIR_PATH).mkdir(exist_ok=True, parents=True)


!ffmpeg -i $vid_path -vf fps=$FPS $IMAGES_DIR_PATH/frame_%04d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### Colmap :: Feature Extraction

In [5]:
# Colmap Feature Extraction
!colmap feature_extractor \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH  --ImageReader.camera_model PINHOLE


Feature extraction

Processed file [1/511]
  Name:            frame_0001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - PINHOLE
  Focal Length:    2304.00px
  Features:        483
Processed file [2/511]
  Name:            frame_0002.jpg
  Dimensions:      1920 x 1080
  Camera:          #2 - PINHOLE
  Focal Length:    2304.00px
  Features:        432
Processed file [3/511]
  Name:            frame_0003.jpg
  Dimensions:      1920 x 1080
  Camera:          #3 - PINHOLE
  Focal Length:    2304.00px
  Features:        399
Processed file [4/511]
  Name:            frame_0004.jpg
  Dimensions:      1920 x 1080
  Camera:          #4 - PINHOLE
  Focal Length:    2304.00px
  Features:        514
Processed file [5/511]
  Name:            frame_0005.jpg
  Dimensions:      1920 x 1080
  Camera:          #5 - PINHOLE
  Focal Length:    2304.00px
  Features:        508
Processed file [6/511]
  Name:            frame_0013.jpg
  Dimensions:      1920 x 1080
  Camera:          #13 - PINHOL

### Colmap :: Feature Matching

In [6]:
# Feature Matching
!colmap sequential_matcher \
    --database_path $DATABASE_PATH


Sequential feature matching

Matching image [1/511] in 0.213s
Matching image [2/511] in 0.077s
Matching image [3/511] in 0.105s
Matching image [4/511] in 0.184s
Matching image [5/511] in 0.136s
Matching image [6/511] in 0.059s
Matching image [7/511] in 0.173s
Matching image [8/511] in 0.150s
Matching image [9/511] in 0.191s
Matching image [10/511] in 0.297s
Matching image [11/511] in 0.025s
Matching image [12/511] in 0.177s
Matching image [13/511] in 0.267s
Matching image [14/511] in 0.094s
Matching image [15/511] in 0.117s
Matching image [16/511] in 0.131s
Matching image [17/511] in 0.125s
Matching image [18/511] in 0.296s
Matching image [19/511] in 0.149s
Matching image [20/511] in 0.083s
Matching image [21/511] in 0.265s
Matching image [22/511] in 0.074s
Matching image [23/511] in 0.138s
Matching image [24/511] in 0.269s
Matching image [25/511] in 0.185s
Matching image [26/511] in 0.178s
Matching image [27/511] in 0.249s
Matching image [28/511] in 0.170s
Matching image [29/511] in 

### Colmap :: Sparse Reconstruction (Mapper)

In [7]:
# Sparse Reconstruction (Mapper)
SPARSE_DIR = os.path.join(DATASET_DIR_PATH, "sparse")
Path(SPARSE_DIR).mkdir(exist_ok=True, parents=True)

!colmap mapper \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_DIR_PATH \
    --output_path $SPARSE_DIR


Loading database

Loading cameras... 511 in 0.001s
Loading matches... 5920 in 0.070s
Loading images... 511 in 0.036s (connected 511)
Building correspondence graph... in 0.245s (ignored 0)

Elapsed time: 0.006 [minutes]


Finding good initial image pair


Initializing with image pair #244 and #500


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  1.908539e+04    0.00e+00    3.00e+05   0.00e+00   0.00e+00  1.00e+04        0    2.57e-04    1.14e-03
   1  1.687548e+05   -1.50e+05    3.00e+05   1.40e+03  -7.91e+00  5.00e+03        1    3.06e-04    1.46e-03
   2  1.128337e+05   -9.37e+04    3.00e+05   1.27e+03  -4.97e+00  1.25e+03        1    1.99e-04    1.67e-03
   3  4.857126e+04   -2.95e+04    3.00e+05   1.02e+03  -1.61e+00  1.56e+02        1    1.95e-04    1.87e-03
   4  9.682562e+03    9.40e+03    4.95e+04   4.74e+02   7.61e-01  1.82e+02        1    3.97e-04    2.27e-03
   5  3.515336e+03    6.1

### Triangle-Splatting :: Training (Indoor mode)

In [8]:
DATASET_DIR_PATH
OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}"
print(DATASET_DIR_PATH)
print(OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTPUR_DIR_PATH -r $RES --eval

./temp_data/Alaska_00
./temp_result/Alaska_00
Optimizing ./temp_result/Alaska_00
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output folder: ./temp_result/Alas

### Triangle-Splatting :: Rendering (Indoor mode)

In [9]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Alaska_00/cfg_args
Config file found: ./temp_result/Alaska_00/cfg_args
Rendering ./temp_result/Alaska_00
Loading trained model at iteration 30000 [04/10 12:57:31]
Reading camera 1/511----- PINHOLE [04/10 12:57:31]
Reading camera 2/511----- PINHOLE [04/10 12:57:31]
Reading camera 3/511----- PINHOLE [04/10 12:57:31]
Reading camera 4/511----- PINHOLE [04/10 12:57:31]
Reading camera 5/511----- PINHOLE [04/10 12:57:31]
Reading camera 6/511----- PINHOLE [04/10 12:57:31]
Reading camera 7/511----- PINHOLE [04/10 12:57:31]
Reading camera 8/511----- PINHOLE [04/10 12:57:31]
Reading camera 9/511----- PINHOLE [04/10 12:57:31]
Reading camera 10/511----- PINHOLE [04/10 12:57:31]
Reading camera 11/511----- PINHOLE [04/10 12:57:31]
Reading camera 12/511----- PINHOLE [04/10 12:57:31]
Reading camera 13/511----- PINHOLE [04/10 12:57:31]
Reading camera 14/511----- PINHOLE [04/10 12:57:31]
Reading camera 15/511----- PINHOLE [04/10 12:57:31]
Reading camera 16/511----

### Triangle-Splatting :: Create a video (Indoor mode)

In [10]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Alaska_00/cfg_args
Config file found: ./temp_result/Alaska_00/cfg_args
Creating video for ./temp_result/Alaska_00
Loading trained model at iteration 30000
Reading camera 1/511----- PINHOLE
Reading camera 2/511----- PINHOLE
Reading camera 3/511----- PINHOLE
Reading camera 4/511----- PINHOLE
Reading camera 5/511----- PINHOLE
Reading camera 6/511----- PINHOLE
Reading camera 7/511----- PINHOLE
Reading camera 8/511----- PINHOLE
Reading camera 9/511----- PINHOLE
Reading camera 10/511----- PINHOLE
Reading camera 11/511----- PINHOLE
Reading camera 12/511----- PINHOLE
Reading camera 13/511----- PINHOLE
Reading camera 14/511----- PINHOLE
Reading camera 15/511----- PINHOLE
Reading camera 16/511----- PINHOLE
Reading camera 17/511----- PINHOLE
Reading camera 18/511----- PINHOLE
Reading camera 19/511----- PINHOLE
Reading camera 20/511----- PINHOLE
Reading camera 21/511----- PINHOLE
Reading camera 22/511----- PINHOLE
Reading camera 23/511----- PINHOLE
Reading 

### Triangle-Splatting :: Training (Outdoor mode)

In [11]:
DATASET_DIR_PATH
OUTDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_outdoor"
print(DATASET_DIR_PATH)
print(OUTDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/train.py -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --eval  --outdoor 

./temp_data/Alaska_00
./temp_result/Alaska_00_outdoor
Optimizing ./temp_result/Alaska_00_outdoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output folder: ./

### Triangle-Splatting :: Rendering (Outdoor mode)

In [12]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/render.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Alaska_00_outdoor/cfg_args
Config file found: ./temp_result/Alaska_00_outdoor/cfg_args
Rendering ./temp_result/Alaska_00_outdoor
Loading trained model at iteration 30000 [04/10 13:28:22]
Reading camera 1/511----- PINHOLE [04/10 13:28:22]
Reading camera 2/511----- PINHOLE [04/10 13:28:22]
Reading camera 3/511----- PINHOLE [04/10 13:28:22]
Reading camera 4/511----- PINHOLE [04/10 13:28:22]
Reading camera 5/511----- PINHOLE [04/10 13:28:22]
Reading camera 6/511----- PINHOLE [04/10 13:28:22]
Reading camera 7/511----- PINHOLE [04/10 13:28:22]
Reading camera 8/511----- PINHOLE [04/10 13:28:22]
Reading camera 9/511----- PINHOLE [04/10 13:28:22]
Reading camera 10/511----- PINHOLE [04/10 13:28:22]
Reading camera 11/511----- PINHOLE [04/10 13:28:22]
Reading camera 12/511----- PINHOLE [04/10 13:28:22]
Reading camera 13/511----- PINHOLE [04/10 13:28:22]
Reading camera 14/511----- PINHOLE [04/10 13:28:22]
Reading camera 15/511----- PINHOLE [04/10 13:28:22]
R

### Triangle-Splatting :: Create a video (Outdoor mode)

In [13]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting/create_video.py -m $OUTDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/Alaska_00_outdoor/cfg_args
Config file found: ./temp_result/Alaska_00_outdoor/cfg_args
Creating video for ./temp_result/Alaska_00_outdoor
Loading trained model at iteration 30000
Reading camera 1/511----- PINHOLE
Reading camera 2/511----- PINHOLE
Reading camera 3/511----- PINHOLE
Reading camera 4/511----- PINHOLE
Reading camera 5/511----- PINHOLE
Reading camera 6/511----- PINHOLE
Reading camera 7/511----- PINHOLE
Reading camera 8/511----- PINHOLE
Reading camera 9/511----- PINHOLE
Reading camera 10/511----- PINHOLE
Reading camera 11/511----- PINHOLE
Reading camera 12/511----- PINHOLE
Reading camera 13/511----- PINHOLE
Reading camera 14/511----- PINHOLE
Reading camera 15/511----- PINHOLE
Reading camera 16/511----- PINHOLE
Reading camera 17/511----- PINHOLE
Reading camera 18/511----- PINHOLE
Reading camera 19/511----- PINHOLE
Reading camera 20/511----- PINHOLE
Reading camera 21/511----- PINHOLE
Reading camera 22/511----- PINHOLE
Reading camera 23/5